# DL GenAI Project: Milestones 1 to 5

**Learner:** Sripathi Dhruv Reddy (`23f3001736`)  
**Competition:** SurfaceGuard AI: Industrial Surface Defect Detection

This notebook follows the guidelines of all milestones: combination-stratified 80/20
split, a native-resolution segmentation baseline trained for five epochs, a
threshold sweep, the Milestone 2 questions, three models compared with Weights & Biases,
in Milestone 4 two longer trained EfficientNet-B3 FPN models, and in Milestone 5 the post-processing checks and the final pipeline.

In [ ]:
from pathlib import Path
import gc, json, os, random

import cv2
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

# seed from my roll number so the split and training can be repeated
SEED = 1736
# images are 128 x 800 grayscale strips, 4 defect classes (one mask channel each)
H, W = 128, 800
CLASSES = ["defect_a", "defect_b", "defect_c", "defect_d"]
# thresholds tried later in the holdout sweep
THRESHOLDS = [0.3, 0.5, 0.7, 0.9]
EPOCHS = 5
BATCH_SIZE = 16

# seed python, numpy and torch (cpu and gpu)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
# benchmark lets cudnn pick the fastest conv algorithm for the fixed input size
torch.backends.cudnn.benchmark = True
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)

## Locate the official competition files

In [ ]:
# find train.csv inside the attached kaggle input (folder name can differ)
roots = list(Path("/kaggle/input").glob("**/public/train.csv"))
if not roots:
    roots = list(Path("/kaggle/input").glob("**/train.csv"))
assert roots, "Official train.csv was not found. Attach the competition data source."
TRAIN_CSV = roots[0]
DATA_ROOT = TRAIN_CSV.parent
TRAIN_DIR = DATA_ROOT / "train"
TEST_DIR = DATA_ROOT / "test"
SAMPLE_CSV = DATA_ROOT / "sample_submission.csv"
print({"train_csv": str(TRAIN_CSV), "train_dir": str(TRAIN_DIR), "test_dir": str(TEST_DIR)})

# keep_default_na=False so an empty rle stays an empty string, not NaN
train_long = pd.read_csv(TRAIN_CSV, keep_default_na=False)
sample = pd.read_csv(SAMPLE_CSV, keep_default_na=False)
# long format: one row per (image, defect_type) pair
assert list(train_long.columns) == ["image_id", "defect_type", "mask_rle", "width", "height"]
assert list(sample.columns) == ["ID", "image_id", "defect_type", "mask_rle"]
print("train rows/images:", len(train_long), train_long.image_id.nunique())
print("test rows/images:", len(sample), sample.image_id.nunique())

## RLE and combination-stratified 80/20 split

Competition RLE uses 1-based starts and column-major (Fortran) flattening.

In [ ]:
def rle_decode(rle, shape=(H, W)):
    # start from a flat zero mask and set each run of 1s
    mask = np.zeros(shape[0] * shape[1], dtype=np.uint8)
    text = str(rle).strip()
    if text:
        # rle is 'start length start length ...', so reshape into pairs
        values = np.asarray(text.split(), dtype=np.int64).reshape(-1, 2)
        for start, length in values:
            # starts are 1-based, so shift by 1
            mask[start - 1:start - 1 + length] = 1
    # column-major: fill down the columns first (order='F')
    return mask.reshape(shape, order="F")


def rle_encode(mask):
    # column-major: flatten the transposed way, same order as the competition rle
    pixels = np.asarray(mask, dtype=np.uint8).reshape(-1, order="F")
    # pad with 0 on both sides so every run has a start and an end
    padded = np.concatenate([[0], pixels, [0]])
    # positions where the value changes, +1 makes them 1-based
    changes = np.where(padded[1:] != padded[:-1])[0] + 1
    # turn every end position into a length (end - start)
    changes[1::2] -= changes[::2]
    return " ".join(map(str, changes))


# wide table: one row per image, one rle column per class
wide = train_long.pivot(index="image_id", columns="defect_type", values="mask_rle").fillna("")
wide = wide.reindex(columns=CLASSES, fill_value="")
# combo string like '0100' tells which classes are present in the image
present = wide.applymap(lambda x: bool(str(x).strip()))
combo = present.astype(int).astype(str).agg("".join, axis=1)
combo_counts = combo.value_counts()
# Singleton combinations cannot be stratified; group only those as "rare".
strata = combo.where(combo.map(combo_counts) >= 2, "rare")
# 80/20 split stratified on the combination so val has the same class mix
train_ids, val_ids = train_test_split(
    wide.index.to_numpy(), test_size=0.20, random_state=SEED, stratify=strata
)
print("split:", len(train_ids), len(val_ids))
print("combination counts:\n", combo.value_counts().sort_index())

In [ ]:
# image file is normally .jpg, fall back to any extension
def find_image(folder, image_id):
    path = folder / f"{image_id}.jpg"
    if path.exists():
        return path
    matches = list(folder.glob(f"{image_id}.*"))
    if not matches:
        raise FileNotFoundError(image_id)
    return matches[0]


# m1 dataset: grayscale image + 4 channel mask, test mode returns only image and id
class SurfaceDataset(Dataset):
    def __init__(self, image_ids, masks=None, augment=False):
        self.ids = list(image_ids)
        self.masks = masks
        self.augment = augment

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        image_id = self.ids[idx]
        # masks given -> train/val images, no masks -> test images
        folder = TRAIN_DIR if self.masks is not None else TEST_DIR
        image = cv2.imread(str(find_image(folder, image_id)), cv2.IMREAD_GRAYSCALE)
        assert image is not None and image.shape == (H, W), (image_id, None if image is None else image.shape)
        # scale to [0, 1] then roughly centre around 0
        image = image.astype(np.float32) / 255.0
        image = (image - 0.5) / 0.25
        if self.masks is None:
            return torch.from_numpy(image[None]), image_id
        # decode the 4 class masks -> shape (4, H, W)
        mask = np.stack([rle_decode(self.masks.loc[image_id, c]) for c in CLASSES]).astype(np.float32)
        # random horizontal flip, same flip on image and mask
        if self.augment and random.random() < 0.5:
            image = np.ascontiguousarray(image[:, ::-1])
            mask = np.ascontiguousarray(mask[:, :, ::-1])
        # image[None] adds the channel dim -> (1, H, W)
        return torch.from_numpy(image[None]), torch.from_numpy(mask), image_id


# drop_last keeps the batch size fixed for batchnorm; val is not shuffled
train_loader = DataLoader(SurfaceDataset(train_ids, wide, True), batch_size=BATCH_SIZE,
                          shuffle=True, num_workers=2, pin_memory=True, drop_last=True)
val_loader = DataLoader(SurfaceDataset(val_ids, wide, False), batch_size=BATCH_SIZE,
                        shuffle=False, num_workers=2, pin_memory=True)

## Native-resolution compact U-Net baseline

In [ ]:
# two 3x3 conv + batchnorm + silu, padding=1 keeps the size
class ConvBlock(nn.Module):
    def __init__(self, cin, cout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(cin, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.SiLU(),
            nn.Conv2d(cout, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.SiLU(),
        )
    def forward(self, x): return self.net(x)


# small 3 level u-net, base=8 channels, channels double at every level
class TinyUNet(nn.Module):
    def __init__(self, base=8):
        super().__init__()
        self.e1, self.e2, self.e3 = ConvBlock(1, base), ConvBlock(base, base*2), ConvBlock(base*2, base*4)
        self.pool = nn.MaxPool2d(2)
        self.mid = ConvBlock(base*4, base*8)
        # decoder: transpose conv doubles H, W, then concat with skip so input has 2x channels
        self.u3 = nn.ConvTranspose2d(base*8, base*4, 2, 2)
        self.d3 = ConvBlock(base*8, base*4)
        self.u2 = nn.ConvTranspose2d(base*4, base*2, 2, 2)
        self.d2 = ConvBlock(base*4, base*2)
        self.u1 = nn.ConvTranspose2d(base*2, base, 2, 2)
        self.d1 = ConvBlock(base*2, base)
        # 1x1 conv -> one logit map per class, output (B, 4, H, W)
        self.out = nn.Conv2d(base, len(CLASSES), 1)
    def forward(self, x):
        # encoder: 128x800 -> 64x400 -> 32x200, middle works at 16x100
        e1 = self.e1(x); e2 = self.e2(self.pool(e1)); e3 = self.e3(self.pool(e2))
        x = self.mid(self.pool(e3))
        # skip connections bring back the fine detail lost by pooling
        x = self.d3(torch.cat([self.u3(x), e3], 1))
        x = self.d2(torch.cat([self.u2(x), e2], 1))
        x = self.d1(torch.cat([self.u1(x), e1], 1))
        return self.out(x)


# dice + bce: bce gives stable pixel gradients, dice handles the small defects
def loss_fn(logits, target):
    bce = nn.functional.binary_cross_entropy_with_logits(logits, target)
    # sigmoid not softmax, classes can overlap so each one is its own yes/no map
    prob = torch.sigmoid(logits)
    # dice per class over batch and pixels, +1 smoothing avoids 0/0 on empty masks
    dims = (0, 2, 3)
    dice = (2 * (prob * target).sum(dims) + 1) / (prob.sum(dims) + target.sum(dims) + 1)
    return bce + (1 - dice.mean())


model = TinyUNet().to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-4)
# mixed precision on gpu: autocast runs in fp16, gradscaler stops small grads becoming 0
scaler = torch.amp.GradScaler("cuda", enabled=DEVICE.type == "cuda")
history = []

# training loop: forward, loss, scaled backward, optimizer step
for epoch in range(1, EPOCHS + 1):
    model.train(); running = 0.0
    for images, masks, _ in train_loader:
        images, masks = images.to(DEVICE, non_blocking=True), masks.to(DEVICE, non_blocking=True)
        # clear old gradients before the new backward
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda", enabled=DEVICE.type == "cuda"):
            loss = loss_fn(model(images), masks)
        # scale loss before backward, scaler.step unscales and skips the step if inf/nan
        scaler.scale(loss).backward()
        scaler.step(optimizer); scaler.update()
        running += loss.item()
    # mean train loss of the epoch
    mean_loss = running / len(train_loader)
    history.append({"epoch": epoch, "train_loss": mean_loss})
    print(f"epoch {epoch}/{EPOCHS} train_loss={mean_loss:.5f}")

# save weights and loss history for the report
torch.save(model.state_dict(), "/kaggle/working/milestone1_tiny_unet.pt")
pd.DataFrame(history).to_csv("/kaggle/working/training_history.csv", index=False)

## Holdout threshold sweep (no minimum-area filtering)

For each threshold I check: (a) how often an empty image/class pair gets a wrong mask,
(b) Dice on pairs that really have a defect, (c) the plain mean Dice over all pairs, and (d) the
competition style score, which also gives marks for getting empty masks right.

In [ ]:
def collect_validation():
    # eval mode (batchnorm uses running stats) and no_grad to save memory
    model.eval(); probs, truths = [], []
    with torch.no_grad():
        for images, masks, _ in val_loader:
            with torch.amp.autocast("cuda", enabled=DEVICE.type == "cuda"):
                p = torch.sigmoid(model(images.to(DEVICE, non_blocking=True)))
            # keep probs as float16 on cpu so the whole val set fits in ram
            probs.append(p.float().cpu().numpy().astype(np.float16))
            truths.append(masks.numpy().astype(np.uint8))
    return np.concatenate(probs), np.concatenate(truths)


# shapes (N, 4, H, W); a pair = one image and one class
val_prob, val_true = collect_validation()
# true_positive: pairs that really have a defect, empty_pair: no defect
true_positive = val_true.sum(axis=(2, 3)) > 0
empty_pair = ~true_positive
rows = []
# for each threshold make binary masks and compute the score parts
for threshold in THRESHOLDS:
    pred = val_prob >= threshold
    pred_positive = pred.sum(axis=(2, 3)) > 0
    intersection = (pred & (val_true > 0)).sum(axis=(2, 3))
    # dice per pair; empty prediction on empty truth counts as dice 1
    denominator = pred.sum(axis=(2, 3)) + val_true.sum(axis=(2, 3))
    pair_dice = np.where(denominator > 0, 2 * intersection / np.maximum(denominator, 1), 1.0)
    # empty pairs: how many get any predicted pixel, and the fraction of wrong pixels
    fpr_empty = pred_positive[empty_pair].mean()
    fp_pixel_fraction = pred[empty_pair].mean()
    false_positive_score = 1.0 - fp_pixel_fraction
    empty_accuracy = 1.0 - fpr_empty
    foreground_dice = pair_dice[true_positive].mean()
    # naive dice averages all pairs, so the many empty pairs push it up
    naive_dice = pair_dice.mean()
    # Official metric: 70% positive-pair Dice, 20% FP-pixel score, 10% empty accuracy.
    official_score = 0.70 * foreground_dice + 0.20 * false_positive_score + 0.10 * empty_accuracy
    rows.append(dict(threshold=threshold, empty_false_positive_rate=fpr_empty,
                     foreground_dice=foreground_dice, naive_mean_dice=naive_dice,
                     fp_pixel_fraction=fp_pixel_fraction,
                     false_positive_score=false_positive_score,
                     empty_accuracy=empty_accuracy, official_score=official_score))

# results table, best threshold by official score and by naive dice
results = pd.DataFrame(rows)
results.to_csv("/kaggle/working/threshold_sweep.csv", index=False)
print(results.to_string(index=False, float_format=lambda x: f"{x:.5f}"))
best_official = float(results.loc[results.official_score.idxmax(), "threshold"])
best_naive = float(results.loc[results.naive_mean_dice.idxmax(), "threshold"])
print("best official threshold:", best_official, "best naive threshold:", best_naive)

# q12 answer text built from the numbers above
first, last = results.iloc[0], results.iloc[-1]
q12 = (
    f"Across tau=0.3 to 0.9, the empty-pair false-positive rate changed from "
    f"{first.empty_false_positive_rate:.4f} to {last.empty_false_positive_rate:.4f}, while positive-pair "
    f"foreground Dice changed from {first.foreground_dice:.4f} to {last.foreground_dice:.4f}. "
    f"The official-style score peaked at tau={best_official:.1f}; naive mean Dice peaked at tau={best_naive:.1f}. "
    "Higher thresholds suppress spurious masks on the many empty image-class pairs, so the naive mean can "
    "prefer a higher tau because true-negative empty pairs dominate it. The official score gives 70% weight "
    "to foreground Dice, 20% to the false-positive pixel score, and 10% to empty accuracy, so it limits that "
    "empty-pair advantage and gives a fairer choice."
)
Path("/kaggle/working/milestone1_q12_answer.txt").write_text(q12)
print("\nQ12 answer:\n", q12)

# free memory before the next part
del val_prob, val_true
gc.collect()
if DEVICE.type == "cuda": torch.cuda.empty_cache()

## Generate the real competition submission

In [ ]:
# one entry per image for the loader, sample has 4 rows (classes) per image
test_ids = sample.image_id.drop_duplicates().tolist()
test_loader = DataLoader(SurfaceDataset(test_ids), batch_size=BATCH_SIZE,
                         shuffle=False, num_workers=2, pin_memory=True)
prediction_by_key = {}
model.eval()
# predict, threshold at best_official, encode each class mask to rle
with torch.no_grad():
    for images, image_ids in test_loader:
        with torch.amp.autocast("cuda", enabled=DEVICE.type == "cuda"):
            probs = torch.sigmoid(model(images.to(DEVICE, non_blocking=True))).float().cpu().numpy()
        for i, image_id in enumerate(image_ids):
            for class_index, defect_type in enumerate(CLASSES):
                prediction_by_key[(image_id, defect_type)] = rle_encode(probs[i, class_index] >= best_official)

# fill mask_rle in the same row order as sample_submission
submission = sample.copy()
submission["mask_rle"] = [prediction_by_key[(i, c)] for i, c in zip(submission.image_id, submission.defect_type)]
# Kaggle's validator expects a literal space for an empty mask, matching sample_submission.csv.
submission["mask_rle"] = submission["mask_rle"].replace("", " ")
# check the columns and ID order match the sample
assert list(submission.columns) == ["ID", "image_id", "defect_type", "mask_rle"]
assert len(submission) == len(sample) and submission.ID.equals(sample.ID)
submission.to_csv("/kaggle/working/submission.csv", index=False)
print("submission rows:", len(submission), "non-empty masks:", (submission.mask_rle.str.len() > 0).sum())
print("Saved /kaggle/working/submission.csv")

---
# Milestone 2: Segmentation pipeline and first improved submission

Everything below reuses the helpers defined above (`rle_decode`, `rle_encode`, `CLASSES`, `TRAIN_DIR`, `wide`).
Competition RLE is 1-based and column-major (`order='F'`).

## Q1. Column-major RLE on the 8 x 8 grid

In [ ]:
# highlighted cells in the question figure (1-based, column-major numbering)
active = [12, 13, 23, 24, 25, 26, 44, 45, 47]
grid = np.zeros((8, 8), dtype=np.uint8)
for idx in active:
    i = idx - 1
    grid[i % 8, i // 8] = 1          # row = i % H, col = i // H
print(grid)
q1 = rle_encode(grid)
print("Q1 RLE:", q1)
assert (rle_decode(q1, (8, 8)) == grid).all()   # round trip check

## Q2. Decoding all four masks of image 0ZUU8LH57Y

In [ ]:
rows = train_long[train_long.image_id == "0ZUU8LH57Y"]
# decode each class mask of this image and count its pixels
per_class = {r.defect_type: int(rle_decode(r.mask_rle).sum()) for r in rows.itertuples()}
print(per_class)
print("Q2 total positive pixels:", sum(per_class.values()))

## Q3. Class distribution and prevalence

In [ ]:
# 1 if the class rle is non-empty, so one 0/1 row per image
present_wide = wide.apply(lambda col: col.str.strip() != "").astype(int)[CLASSES]
counts = present_wide.sum()
# prevalence = % of images that contain the class
summary = pd.DataFrame({"images": counts, "prevalence_%": (100 * counts / len(present_wide)).round(2)})
print("rows:", len(train_long), "images:", len(present_wide))
print(summary)
print("clean images (no defect):", int((present_wide.sum(1) == 0).sum()))
print("Q3:", f"{counts.idxmax()} ({counts.max()}), {counts.idxmin()} ({counts.min()})")

## Q4. Co-occurrence matrix C = X^T X

In [ ]:
# X is (images, 4) of 0/1, so X^T X gives pair counts; diagonal = class counts
X = present_wide.values
C = pd.DataFrame(X.T @ X, index=CLASSES, columns=CLASSES)
print(C)
# drop the diagonal and find the most frequent pair
off = C.where(~np.eye(4, dtype=bool)).stack()
pair = off.idxmax()
print("Q4:", pair, int(off.max()))

## Q5. Co-occurrence vs correlation (phi coefficient)

In [ ]:
# pearson correlation on 0/1 columns is the phi coefficient
corr = present_wide.corr()
print(corr.round(4))
n = len(present_wide)
# compare observed co-occurrence with the count expected if classes were independent
for i in range(4):
    for j in range(i + 1, 4):
        a, b = CLASSES[i], CLASSES[j]
        expected = counts[a] * counts[b] / n
        print(f"{a}-{b}: observed {C.loc[a, b]}, expected if independent {expected:.1f}, phi {corr.loc[a, b]:.4f}")
print("Q5:", round(corr.loc[pair[0], pair[1]], 2))

## Q6. Pixel-level overlap between defect classes

In [ ]:
# only images with 2+ defect classes can overlap
multi_ids = present_wide.index[present_wide.sum(1) >= 2]
max_overlap, max_dice, checked = 0, 0.0, 0
for image_id in multi_ids:
    masks_here = [rle_decode(wide.loc[image_id, c]) for c in CLASSES if present_wide.loc[image_id, c]]
    for i in range(len(masks_here)):
        for j in range(i + 1, len(masks_here)):
            # overlap pixels and dice between two class masks of the same image
            inter = int((masks_here[i] & masks_here[j]).sum())
            dice = 2 * inter / (masks_here[i].sum() + masks_here[j].sum())
            max_overlap, max_dice, checked = max(max_overlap, inter), max(max_dice, dice), checked + 1
print("multi-label images:", len(multi_ids), "pairs checked:", checked)
print("Q6 max overlap pixels:", max_overlap, "max pairwise Dice:", max_dice)

## Q7. PyTorch Dataset for Paradigm 1 (RGB image, 4-channel float32 mask)

`__getitem__` returns a tuple `(image, masks)`: image `[3, 128, 800]` in [0, 1], masks `[4, 128, 800]` float32.

In [ ]:
# m2 dataset: rgb image (3, H, W) in [0, 1] and 4 float32 masks (4, H, W)
class SurfaceAnomalyDataset(Dataset):
    def __init__(self, image_df, image_dir, augment=False):
        self.df = image_df.reset_index(drop=True)
        self.image_dir = Path(image_dir)
        self.augment = augment

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = cv2.imread(str(self.image_dir / f"{row['image_id']}.jpg"), cv2.IMREAD_COLOR)
        # cv2 reads bgr, convert to rgb
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
        masks = np.stack([rle_decode(row[c]) for c in CLASSES]).astype(np.float32)
        # hwc -> chw for pytorch
        image = image.transpose(2, 0, 1)
        # same random flips on image and masks (axis 2 = width, axis 1 = height)
        if self.augment:
            if random.random() < 0.5:
                image, masks = image[:, :, ::-1], masks[:, :, ::-1]
            if random.random() < 0.5:
                image, masks = image[:, ::-1, :], masks[:, ::-1, :]
        # flips give negative strides, make contiguous before torch.from_numpy
        image = torch.from_numpy(np.ascontiguousarray(image))
        masks = torch.from_numpy(np.ascontiguousarray(masks))
        return image, masks


# check one batch: shapes, dtype and value range
wide_df = wide.reset_index()
check_ds = SurfaceAnomalyDataset(wide_df, TRAIN_DIR)
images, masks = next(iter(DataLoader(check_ds, batch_size=4, shuffle=False)))
print("dataset length:", len(check_ds))
print("images:", list(images.shape), images.dtype, "range", float(images.min()), float(images.max()))
print("masks:", list(masks.shape), masks.dtype)

## Q8. Baseline 3-stage U-Net, 5 epochs, AdamW lr 3e-4, BCEWithLogitsLoss

In [ ]:
BASE_EPOCHS = 5
# same image ids as the m1 split
tr_df = wide_df[wide_df.image_id.isin(train_ids)]
va_df = wide_df[wide_df.image_id.isin(val_ids)]
base_train_loader = DataLoader(SurfaceAnomalyDataset(tr_df, TRAIN_DIR), batch_size=16, shuffle=True,
                               num_workers=4, pin_memory=True, drop_last=True)
base_val_loader = DataLoader(SurfaceAnomalyDataset(va_df, TRAIN_DIR), batch_size=16, shuffle=False,
                             num_workers=4, pin_memory=True)


# conv-bn-relu twice, padding=1 keeps H and W
def block(cin, cout):
    return nn.Sequential(nn.Conv2d(cin, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
                         nn.Conv2d(cout, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(inplace=True))


# 3 stage u-net: 3 poolings, so H and W must divide by 8 (128 and 800 do)
class UNet3(nn.Module):
    def __init__(self, b=16):
        super().__init__()
        self.e1, self.e2, self.e3 = block(3, b), block(b, 2 * b), block(2 * b, 4 * b)
        self.pool = nn.MaxPool2d(2)
        self.mid = block(4 * b, 8 * b)
        self.u3, self.d3 = nn.ConvTranspose2d(8 * b, 4 * b, 2, 2), block(8 * b, 4 * b)
        self.u2, self.d2 = nn.ConvTranspose2d(4 * b, 2 * b, 2, 2), block(4 * b, 2 * b)
        self.u1, self.d1 = nn.ConvTranspose2d(2 * b, b, 2, 2), block(2 * b, b)
        # 4 output logits, one per class
        self.out = nn.Conv2d(b, 4, 1)

    def forward(self, x):
        # encoder with pooling, then decoder with skip connections
        e1 = self.e1(x); e2 = self.e2(self.pool(e1)); e3 = self.e3(self.pool(e2))
        m = self.mid(self.pool(e3))
        x = self.d3(torch.cat([self.u3(m), e3], 1))
        x = self.d2(torch.cat([self.u2(x), e2], 1))
        x = self.d1(torch.cat([self.u1(x), e1], 1))
        return self.out(x)


# baseline as asked: adamw lr 3e-4, plain bce with logits (sigmoid is inside the loss)
torch.manual_seed(SEED)
base_model = UNet3().to(DEVICE)
base_opt = torch.optim.AdamW(base_model.parameters(), lr=3e-4)
criterion = nn.BCEWithLogitsLoss()
base_history = []
for epoch in range(1, BASE_EPOCHS + 1):
    base_model.train(); total, batches = 0.0, 0
    # forward, loss, backward, update
    for x, y in base_train_loader:
        x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
        base_opt.zero_grad(set_to_none=True)
        loss = criterion(base_model(x), y)
        loss.backward(); base_opt.step()
        total += loss.item(); batches += 1
    # validation loss averaged per image
    base_model.eval(); vtotal, vcount = 0.0, 0
    with torch.no_grad():
        for x, y in base_val_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            vtotal += criterion(base_model(x), y).item() * x.size(0); vcount += x.size(0)
    base_history.append({"epoch": epoch, "train_loss": total / batches, "val_loss": vtotal / vcount})
    print(f"epoch {epoch}: train_loss={total / batches:.4f} val_loss={vtotal / vcount:.4f}")
    # save the weights only when val loss is the best so far
    if base_history[-1]["val_loss"] == min(h["val_loss"] for h in base_history):
        torch.save(base_model.state_dict(), "/kaggle/working/m2_baseline_best.pt")

base_history = pd.DataFrame(base_history)
base_history.to_csv("/kaggle/working/m2_baseline_history.csv", index=False)
print(base_history)
print("best epoch:", int(base_history.loc[base_history.val_loss.idxmin(), "epoch"]))

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(6, 3.5))
# train vs val loss per epoch, a growing gap would mean overfitting
plt.plot(base_history.epoch, base_history.train_loss, marker="o", label="train")
plt.plot(base_history.epoch, base_history.val_loss, marker="o", label="validation")
plt.xlabel("epoch"); plt.ylabel("BCE loss"); plt.title("Q8 baseline U-Net loss"); plt.legend()
plt.tight_layout(); plt.show()

---
# Milestone 3: three models compared with Weights & Biases

The project needs at least three models and a W&B run for each one:
- **Model A, built from scratch:** my own 3 stage U-Net (wider version of the Q8 baseline)
- **Model B, pretrained:** U-Net with an ImageNet pretrained ResNet34 encoder
- **Model C, my choice:** FPN decoder with an ImageNet pretrained EfficientNet-B2 encoder

All three use the same split, loss (BCE + Dice), flips as augmentation and the same validation metrics,
so the W&B runs in project `23f3001736-t32026` can be compared directly.

In [ ]:
import os, subprocess, sys, time
# smp gives pretrained encoders, install it if missing
try:
    import segmentation_models_pytorch as smp
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "segmentation-models-pytorch"], check=False)
    import segmentation_models_pytorch as smp
import wandb
from sklearn.metrics import f1_score

WANDB_PROJECT = "23f3001736-t32026"
os.environ["WANDB_DIR"] = "/kaggle/working"
# w&b login with the kaggle secret, else log offline
try:
    from kaggle_secrets import UserSecretsClient
    wandb.login(key=UserSecretsClient().get_secret("WANDB_API_KEY"))
    WANDB_MODE = "online"
except Exception as err:
    # no key here: runs are saved offline in /kaggle/working/wandb and uploaded later with `wandb sync`
    print("W&B key not found, logging offline:", err)
    WANDB_MODE = "offline"
try:
    # my W&B team (project 23f3001736-t32026 lives there)
    WANDB_ENTITY = UserSecretsClient().get_secret("WANDB_ENTITY")
except Exception:
    WANDB_ENTITY = "23f3001736-indian-institute-of-technology-madras"
print("W&B mode:", WANDB_MODE, "| entity:", WANDB_ENTITY)

# more epochs for the pretrained models B and C
A_EPOCHS, B_EPOCHS, C_EPOCHS = 15, 25, 25
amp_on = DEVICE.type == "cuda"
# imagenet mean/std, shape (1, 3, 1, 1) to broadcast over (B, 3, H, W)
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)


class Normalized(nn.Module):
    # the pretrained encoders expect ImageNet normalised input
    def __init__(self, net):
        super().__init__()
        self.net = net
        self.register_buffer("mean", MEAN.clone())
        self.register_buffer("std", STD.clone())

    def forward(self, x):
        return self.net((x - self.mean) / self.std)


# same bce + dice loss as milestone 1
def bce_dice(logits, target):
    bce = nn.functional.binary_cross_entropy_with_logits(logits, target)
    prob = torch.sigmoid(logits)
    dims = (0, 2, 3)
    dice = (2 * (prob * target).sum(dims) + 1) / (prob.sum(dims) + target.sum(dims) + 1)
    return bce + (1 - dice.mean())


# train loader with hflip + vflip augmentation, val without
m3_train_loader = DataLoader(SurfaceAnomalyDataset(tr_df, TRAIN_DIR, augment=True), batch_size=16, shuffle=True,
                             num_workers=4, pin_memory=True, drop_last=True)
m3_val_loader = DataLoader(SurfaceAnomalyDataset(va_df, TRAIN_DIR), batch_size=16, shuffle=False,
                           num_workers=4, pin_memory=True)

In [ ]:
def evaluate(model, loader):
    # val loss, Dice on pairs with a defect, accuracy and macro F1 of 'is this defect present'
    model.eval()
    total, count, inter, pred_px, gt_px = 0.0, 0, [], [], []
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            # autocast for speed, .float() before the loss
            with torch.amp.autocast("cuda", enabled=amp_on):
                logits = model(x).float()
            total += bce_dice(logits, y).item() * x.size(0)
            count += x.size(0)
            # per (image, class) counts: overlap, predicted pixels, true pixels
            pred = torch.sigmoid(logits) > 0.5
            inter.append((pred & (y > 0.5)).sum((2, 3)).cpu())
            pred_px.append(pred.sum((2, 3)).cpu())
            gt_px.append(y.sum((2, 3)).cpu())
    inter, pred_px, gt_px = torch.cat(inter).numpy(), torch.cat(pred_px).numpy(), torch.cat(gt_px).numpy()
    # dice only on pairs with a defect; accuracy and f1 on 'defect present or not'
    has_defect, said_defect = gt_px > 0, pred_px > 0
    dice = 2 * inter / np.maximum(pred_px + gt_px, 1)
    return {
        "val_loss": total / count,
        "val_dice": float(dice[has_defect].mean()),
        "val_accuracy": float((has_defect == said_defect).mean()),
        "val_f1": float(np.mean([f1_score(has_defect[:, c], said_defect[:, c], zero_division=0) for c in range(4)])),
    }


def train_model(name, model, epochs, lr, notes):
    # one w&b run per model with its settings
    run = wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY, name=name, mode=WANDB_MODE, reinit=True,
                     config={"model": name, "epochs": epochs, "lr": lr, "batch_size": 16, "loss": "bce+dice",
                             "augment": "hflip+vflip", "notes": notes})
    model = model.to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    # onecycle: lr warms up for 10% of steps then anneals, stepped every batch
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=epochs * len(m3_train_loader), pct_start=0.1)
    scaler = torch.amp.GradScaler("cuda", enabled=amp_on)
    best, best_path, history = -1.0, f"/kaggle/working/m3_{name}.pt", []
    for epoch in range(1, epochs + 1):
        start = time.time()
        model.train()
        running, batches = 0.0, 0
        for x, y in m3_train_loader:
            # forward under autocast, scaled backward, optimizer step, then scheduler step
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", enabled=amp_on):
                loss = bce_dice(model(x), y)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            sched.step()
            running += loss.item()
            batches += 1
        # validate each epoch and log to w&b
        metrics = {"epoch": epoch, "train_loss": running / batches, **evaluate(model, m3_val_loader),
                   "lr": sched.get_last_lr()[0]}
        history.append(metrics)
        wandb.log(metrics)
        print(f"{name} epoch {epoch}: train {metrics['train_loss']:.4f} val {metrics['val_loss']:.4f} "
              f"dice {metrics['val_dice']:.4f} acc {metrics['val_accuracy']:.4f} f1 {metrics['val_f1']:.4f} "
              f"({time.time() - start:.0f}s)", flush=True)
        # keep the checkpoint with the best val dice
        if metrics["val_dice"] > best:
            best = metrics["val_dice"]
            torch.save(model.state_dict(), best_path)
    # reload the best epoch weights before returning
    model.load_state_dict(torch.load(best_path, map_location=DEVICE))
    best_row = max(history, key=lambda h: h["val_dice"])
    for k, v in best_row.items():
        run.summary["best_" + k] = v
    run.finish()
    pd.DataFrame(history).to_csv(f"/kaggle/working/m3_{name}_history.csv", index=False)
    return model, best_row

### Model A: U-Net built from scratch

In [ ]:
torch.manual_seed(SEED)
# model A: wider u-net (b=32), random init, so a higher lr 1e-3
model_a, best_a = train_model("A_scratch_unet", Normalized(UNet3(b=32)), A_EPOCHS, 1e-3,
                              "3 stage U-Net, random init, no pretrained weights")

### Model B: pretrained ResNet34 U-Net

In [ ]:
torch.manual_seed(SEED)
# model B: smp u-net with imagenet resnet34 encoder, lower lr for pretrained weights
model_b, best_b = train_model("B_resnet34_unet",
                              Normalized(smp.Unet("resnet34", encoder_weights="imagenet", in_channels=3, classes=4)),
                              B_EPOCHS, 3e-4, "U-Net decoder, ImageNet ResNet34 encoder")

### Model C: FPN with a pretrained EfficientNet-B2 encoder

In [ ]:
torch.manual_seed(SEED)
# fall back to resnet34 if the effnet-b2 weights cannot be loaded
try:
    net_c, c_name = smp.FPN("efficientnet-b2", encoder_weights="imagenet", in_channels=3, classes=4), "C_effb2_fpn"
except Exception as err:
    print("EfficientNet-B2 weights not available, using ResNet34 FPN instead:", err)
    net_c, c_name = smp.FPN("resnet34", encoder_weights="imagenet", in_channels=3, classes=4), "C_resnet34_fpn"
model_c, best_c = train_model(c_name, Normalized(net_c), C_EPOCHS, 3e-4, "FPN decoder, ImageNet encoder")

### Comparing the three models (same validation split, best epoch of each)

In [ ]:
import matplotlib.pyplot as plt
# best-epoch metrics of each model side by side
compare = pd.DataFrame([best_a, best_b, best_c], index=["A scratch U-Net", "B ResNet34 U-Net", "C " + c_name])
compare = compare[["epoch", "train_loss", "val_loss", "val_dice", "val_accuracy", "val_f1"]]
print(compare.round(4))
compare.to_csv("/kaggle/working/m3_model_comparison.csv")
compare[["val_dice", "val_accuracy", "val_f1"]].plot(kind="bar", figsize=(8, 4), rot=0, ylim=(0, 1))
plt.title("Validation metrics of the three models"); plt.tight_layout()
plt.savefig("/kaggle/working/m3_model_comparison.png", dpi=150); plt.show()

### Ensemble, threshold and minimum area tuning on validation

Probabilities from hflip test time augmentation are averaged over the chosen models. For each class I pick a
threshold and a minimum mask area that give the best competition style score on the validation split.

In [ ]:
# grid of thresholds and min mask areas to tune per class
TAUS = [0.3, 0.4, 0.5, 0.6, 0.7]
MIN_AREAS = [0, 100, 250, 500, 1000, 1500, 2000, 3000, 4000, 6000]
models = {"A": model_a, "B": model_b, "C": model_c}
# model combinations to compare
COMBOS = {"B": ["B"], "C": ["C"], "B+C": ["B", "C"], "A+B+C": ["A", "B", "C"]}


# hflip tta: predict on the flipped image, flip the prediction back, average
def tta_probs(model, x):
    with torch.no_grad(), torch.amp.autocast("cuda", enabled=amp_on):
        p = torch.sigmoid(model(x)).float()
        p_flip = torch.sigmoid(model(torch.flip(x, dims=[3]))).float()
    return (p + torch.flip(p_flip, dims=[3])) / 2


# for every val batch store predicted pixel count and overlap for each tau
for m in models.values():
    m.eval()
stats = {k: {"pc": [], "ic": []} for k in COMBOS}
gt_count = []
for x, y in m3_val_loader:
    x, y = x.to(DEVICE), y.to(DEVICE)
    probs = {k: tta_probs(m, x) for k, m in models.items()}
    for combo, members in COMBOS.items():
        # ensemble = mean of the members' probabilities
        p = sum(probs[k] for k in members) / len(members)
        pc, ic = [], []
        for t in TAUS:
            pm = p >= t
            pc.append(pm.sum((2, 3)))
            ic.append((pm & (y > 0.5)).sum((2, 3)))
        stats[combo]["pc"].append(torch.stack(pc, -1).cpu())
        stats[combo]["ic"].append(torch.stack(ic, -1).cpu())
    gt_count.append(y.sum((2, 3)).cpu())
gt_count = torch.cat(gt_count).numpy()
# positive = pair has a defect (used for dice), the rest are empty pairs
positive = gt_count > 0
n_pos, n_empty = positive.sum(), (~positive).sum()


# for each class try every (tau, min_area) and keep the best
def tune(pred_count, inter_count):
    cfg, total = {}, 0.0
    for c, name in enumerate(CLASSES):
        options = []
        for ti, tau in enumerate(TAUS):
            for area in MIN_AREAS:
                # min area: the mask is dropped (empty) if it has fewer pixels than area
                keep = pred_count[:, c, ti] >= max(area, 1)
                pos = positive[:, c]
                dice = np.where(keep, 2 * inter_count[:, c, ti] / np.maximum(pred_count[:, c, ti] + gt_count[:, c], 1), 0.0)
                # 0.7 dice on defect pairs + 0.3 for keeping empty pairs empty (simplified score)
                score = 0.7 * dice[pos].sum() / n_pos + 0.3 * (~keep[~pos]).sum() / n_empty
                options.append((score, tau, area))
        cfg[name] = max(options)
        total += cfg[name][0]
    return total, cfg


# tune each combo and pick the best one
combo_scores = {}
for combo in COMBOS:
    pcount = torch.cat(stats[combo]["pc"]).numpy()
    icount = torch.cat(stats[combo]["ic"]).numpy()
    combo_scores[combo] = tune(pcount, icount)
    print(f"{combo}: tuned validation score {combo_scores[combo][0]:.4f}")
best_combo = max(combo_scores, key=lambda k: combo_scores[k][0])
best_cfg = combo_scores[best_combo][1]
print("best combo:", best_combo)
for name in CLASSES:
    print(name, "threshold", best_cfg[name][1], "min_area", best_cfg[name][2])

# log the comparison table and tuned scores to a summary w&b run
summary_run = wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY, name="model_comparison", mode=WANDB_MODE, reinit=True)
table = wandb.Table(columns=["model", "best_epoch", "val_loss", "val_dice", "val_accuracy", "val_f1"])
for label, row in compare.iterrows():
    table.add_data(label, int(row["epoch"]), row["val_loss"], row["val_dice"], row["val_accuracy"], row["val_f1"])
wandb.log({"model_comparison": table,
           **{f"tuned_score_{k}": v[0] for k, v in combo_scores.items()}, "best_combo": best_combo})
summary_run.finish()

---
# Milestone 3 assessment: benchmark under the official protocol

Questions 2, 8 and 9 of the milestone need a controlled comparison: seed 42, 5 epochs, grayscale input in [0, 1], no augmentation, no scheduler, batch size 16 and threshold 0.5.
The answers in my form came from the same code run in my separate notebook `dl-23f3001736-milestone3-run`, so the numbers here can be a little different.

In [ ]:
import platform

# free the GPU memory used by the earlier models
gc.collect()
torch.cuda.empty_cache()

# benchmark protocol settings: lr, weight decay, epochs, batch size
BM_LR, BM_WEIGHT_DECAY, BM_EPOCHS, BM_BATCH = 3e-4, 1e-2, 5, 16


class StripDataset(Dataset):
    # grayscale float32 in [0, 1], no augmentation and no extra normalisation
    def __init__(self, ids):
        self.ids = list(ids)

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        image_id = self.ids[idx]
        image = cv2.imread(str(TRAIN_DIR / f"{image_id}.jpg"), cv2.IMREAD_GRAYSCALE).astype(np.float32) / 255.0
        mask = np.stack([rle_decode(wide.loc[image_id, c]) for c in CLASSES]).astype(np.float32)
        return torch.from_numpy(image[None]), torch.from_numpy(mask)


# soft dice over the whole flattened batch, smooth avoids 0/0
class SoftDiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth

    def forward(self, logits, targets):
        probs_flat = torch.sigmoid(logits).reshape(-1)
        targets_flat = targets.reshape(-1)
        intersection = (probs_flat * targets_flat).sum()
        dice = (2.0 * intersection + self.smooth) / (probs_flat.sum() + targets_flat.sum() + self.smooth)
        return 1.0 - dice


# 0.5 bce + 0.5 dice
class CompoundLoss(nn.Module):
    def __init__(self, alpha=0.5, beta=0.5, smooth=1.0):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()
        self.dice = SoftDiceLoss(smooth=smooth)
        self.alpha = alpha
        self.beta = beta

    def forward(self, logits, targets):
        return self.alpha * self.bce(logits, targets) + self.beta * self.dice(logits, targets)


# smp u-net, 1 input channel (grayscale), raw logits out (activation=None)
def bm_model(encoder, weights):
    # seed 42 before building, as the protocol says
    torch.manual_seed(42)
    return smp.Unet(encoder_name=encoder, encoder_weights=weights, in_channels=1, classes=4, encoder_depth=5,
                    decoder_channels=(256, 128, 64, 32, 16), decoder_use_norm="batchnorm",
                    decoder_interpolation="nearest", activation=None)

## Q2. Parameter audit

In [ ]:
bm_resnet = bm_model("resnet34", None)
bm_effnet = bm_model("efficientnet-b0", None)
# total parameters of each model
res_total = sum(p.numel() for p in bm_resnet.parameters())
eff_total = sum(p.numel() for p in bm_effnet.parameters())
# freeze the encoder and count what is still trainable
for p in bm_resnet.encoder.parameters():
    p.requires_grad = False
res_trainable = sum(p.numel() for p in bm_resnet.parameters() if p.requires_grad)
print("resnet34 total:", res_total, " efficientnet-b0 total:", eff_total, " difference:", res_total - eff_total)
print("resnet34 trainable with frozen encoder (%):", round(100 * res_trainable / res_total, 1))
print("ratio:", round(res_total / eff_total, 3))
print("versions:", platform.python_version(), torch.__version__, smp.__version__, torch.version.cuda)
del bm_resnet, bm_effnet

## Q9. Peak training memory (fresh process for each model, on the same GPU)

In [ ]:
# save one fixed batch so both models are measured on the same input
x0, y0 = next(iter(DataLoader(StripDataset(train_ids), batch_size=BM_BATCH, shuffle=False, num_workers=0)))
torch.save({"x": x0, "y": y0}, "/kaggle/working/mem_batch.pt")
# script runs in a new python process, so memory used by this notebook does not count
MEM_CODE = r'''
import sys, torch, torch.nn as nn, segmentation_models_pytorch as smp
encoder = sys.argv[1]
b = torch.load("/kaggle/working/mem_batch.pt")
x, y = b["x"].cuda(), b["y"].cuda()
torch.manual_seed(42)
model = smp.Unet(encoder_name=encoder, encoder_weights="imagenet", in_channels=1, classes=4, encoder_depth=5,
                 decoder_channels=(256, 128, 64, 32, 16), decoder_use_norm="batchnorm",
                 decoder_interpolation="nearest", activation=None).cuda()
bce = nn.BCEWithLogitsLoss()
def crit(l, t):
    p = torch.sigmoid(l).reshape(-1); f = t.reshape(-1)
    return 0.5 * bce(l, t) + 0.5 * (1 - (2 * (p * f).sum() + 1) / (p.sum() + f.sum() + 1))
opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-2)
model.train()
def step():
    opt.zero_grad(); loss = crit(model(x), y); loss.backward(); opt.step()
step(); step()
torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats()
step()
torch.cuda.synchronize()
print("PEAK", round(torch.cuda.max_memory_allocated() / 2**20, 2))
'''
# run the script once per encoder and read the PEAK line
with open("/kaggle/working/mem.py", "w") as f:
    f.write(MEM_CODE)
for enc in ["resnet34", "efficientnet-b0"]:
    r = subprocess.run([sys.executable, "/kaggle/working/mem.py", enc], capture_output=True, text=True)
    line = [l for l in r.stdout.splitlines() if l.startswith("PEAK")]
    print(enc, "peak MiB:", line[0].split()[1] if line else ("oom" if "out of memory" in r.stderr else r.stderr[-300:]))
os.remove("/kaggle/working/mem_batch.pt")

## Q8 and Q9. Five epoch runs

In [ ]:
def bm_eval(model, ids, criterion, metrics=False):
    # eval mode, fixed order, loss weighted by batch size, hard masks at 0.5 with no area filter
    model.eval()
    # fixed order loader, no shuffle
    loader = DataLoader(StripDataset(ids), batch_size=BM_BATCH, shuffle=False, num_workers=0, drop_last=False)
    loss_sum, seen = 0.0, 0
    dice_sum, n_pos, fp_sum, empty_ok, n_empty = 0.0, 0, 0.0, 0, 0
    with torch.inference_mode():
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            # forward, add loss weighted by batch size
            logits = model(x)
            loss_sum += criterion(logits, y).item() * x.size(0)
            seen += x.size(0)
            if metrics:
                # hard masks at 0.5, then per (image, class) pixel counts
                pred = torch.sigmoid(logits) >= 0.5
                truth = y > 0.5
                inter = (pred & truth).sum((2, 3)).float()
                pred_px = pred.sum((2, 3)).float()
                true_px = truth.sum((2, 3)).float()
                pos = true_px > 0
                # dice only on pairs with a defect
                dice = 2 * inter / (pred_px + true_px).clamp(min=1)
                dice_sum += dice[pos].sum().item()
                n_pos += pos.sum().item()
                # empty pairs: fraction of wrong pixels and whether the mask stayed empty
                fp_sum += (pred_px[~pos] / (H * W)).sum().item()
                empty_ok += (pred_px[~pos] == 0).sum().item()
                n_empty += (~pos).sum().item()
    out = {"loss": loss_sum / seen}
    if metrics:
        out["foreground_dice"] = dice_sum / n_pos
        out["false_positive_score"] = 1 - fp_sum / n_empty
        out["empty_accuracy"] = empty_ok / n_empty
        # official mix: 0.7 dice + 0.2 fp score + 0.1 empty accuracy
        out["score"] = 0.7 * out["foreground_dice"] + 0.2 * out["false_positive_score"] + 0.1 * out["empty_accuracy"]
    return out


def bm_train(name, model):
    # reset all seeds and the loader generator so each run is repeatable
    random.seed(42)
    np.random.seed(42)
    torch.manual_seed(42)
    torch.cuda.manual_seed_all(42)
    gen = torch.Generator()
    gen.manual_seed(42)
    model = model.to(DEVICE)
    criterion = CompoundLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=BM_LR, weight_decay=BM_WEIGHT_DECAY)
    loader = DataLoader(StripDataset(train_ids), batch_size=BM_BATCH, shuffle=True, num_workers=0,
                        drop_last=False, generator=gen)
    rows = []
    # plain training: no amp, no scheduler, no augmentation
    for epoch in range(1, BM_EPOCHS + 1):
        model.train()
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(x), y)
            loss.backward()
            optimizer.step()
        # after each epoch eval on train and val in eval mode, gap = |val - train|
        tr = bm_eval(model, train_ids, criterion)
        va = bm_eval(model, val_ids, criterion, metrics=True)
        rows.append({"model": name, "epoch": epoch, "train_eval_loss": tr["loss"], "val_eval_loss": va["loss"],
                     "gap": abs(va["loss"] - tr["loss"]), "foreground_dice": va["foreground_dice"], "score": va["score"]})
        print(rows[-1], flush=True)
    # free gpu memory before the next model
    del model, optimizer
    gc.collect()
    torch.cuda.empty_cache()
    return rows

In [ ]:
# same decoder and head start for the two resnet models, only the encoder start changes
bm_scratch = bm_model("resnet34", None)
bm_pretrained = bm_model("resnet34", "imagenet")
bm_pretrained.decoder.load_state_dict(bm_scratch.decoder.state_dict())
bm_pretrained.segmentation_head.load_state_dict(bm_scratch.segmentation_head.state_dict())

# train the three models one after the other
bm_rows = bm_train("resnet34_scratch", bm_scratch)
bm_rows += bm_train("resnet34_pretrained", bm_pretrained)
bm_rows += bm_train("efficientnet_b0_pretrained", bm_model("efficientnet-b0", "imagenet"))
del bm_scratch, bm_pretrained

bm_table = pd.DataFrame(bm_rows)
bm_table.to_csv("/kaggle/working/m3_benchmark_history.csv", index=False)
print(bm_table[bm_table.epoch.isin([1, 5])].round(4).to_string(index=False))

---
# Milestone 4 assessment: imbalance, schedules and controlled experiments

Working subset: the first 1,200 image ids in sorted order, with an 80/20 split stratified on the combination string (960 train, 240 validation).
Only one image falls in `rare_other`, which cannot be stratified, so it goes to train and the other 1,199 images are stratified.
The answers in my form came from the same code in my separate notebook `dl-23f3001736-milestone45-run`, so the training numbers here can be a little different.

In [ ]:
import math
from torch.utils.data import WeightedRandomSampler

CANVAS = H * W
ms4_df = train_long.copy()   # keep the table, this name is used for a function later


def ms4_decode(rle):
    # same as rle_decode, but runs past the canvas are clipped
    mask = np.zeros(CANVAS, dtype=np.uint8)
    text = str(rle).strip()
    if text:
        v = np.asarray(text.split(), dtype=np.int64).reshape(-1, 2)
        for start, length in v:
            mask[start - 1:min(start - 1 + length, CANVAS)] = 1
    return mask.reshape((H, W), order="F")


# working subset: first 1200 ids in sorted order
ms4_ids = sorted(ms4_df.image_id.unique())[:1200]
ms4_sub = ms4_df[ms4_df.image_id.isin(ms4_ids)]
# wide table and combination string per image ('clean' if no defect)
ms4_wide = ms4_sub.pivot(index="image_id", columns="defect_type", values="mask_rle").reindex(index=ms4_ids, columns=CLASSES).fillna("")
ms4_present = ms4_wide.apply(lambda c: c.str.strip() != "")
ms4_combo = ms4_present.apply(lambda r: "+".join(c for c in CLASSES if r[c]) or "clean", axis=1)
ms4_counts = ms4_combo.value_counts()
print("pairs:", len(ms4_sub))
print(ms4_counts)

## Q1. Image and class level imbalance

In [ ]:
print("clean images:", (ms4_combo == "clean").sum())
print("positive images per class:", ms4_present.sum().to_dict())
print("combination strings:", ms4_counts.size, " largest:", ms4_counts.max(), " smallest:", ms4_counts.min())

## Q2. Pixel level imbalance of defect_b

In [ ]:
# defect_b pixel area of every image (0 if none)
areas_b = np.array([ms4_decode(r).sum() for r in ms4_wide["defect_b"]])
pos_b = areas_b.sum()
# positive % over all pixels of the 1200 images, then negative:positive ratio
print("positive pixels (%):", round(100 * pos_b / (1200 * CANVAS), 4))
print("mean area of positive images:", round(areas_b[areas_b > 0].mean(), 1))
print("negative to positive ratio:", round((1200 * CANVAS - pos_b) / pos_b))

## Q3. Bounded square root sampling

In [ ]:
# weight per combination group = sqrt(n_clean / n_group), capped at 5
n_clean = ms4_counts["clean"]
wg = {g: min(math.sqrt(n_clean / n), 5) for g, n in ms4_counts.items()}
# share of draws that go to groups containing defect_b
total_w = sum(wg[g] * n for g, n in ms4_counts.items())
b_groups = [g for g in ms4_counts.index if "defect_b" in g]
p_b = sum(wg[g] * ms4_counts[g] for g in b_groups) / total_w
# natural share without the sampler, enrichment = sampler share / natural share
natural_b = sum(ms4_counts[g] for g in b_groups) / 1200
print("clean share, sampler and natural (%):", round(100 * wg["clean"] * n_clean / total_w, 2), round(100 * n_clean / 1200, 2))
print("defect_b draws, sampler and natural:", round(1200 * p_b, 2), round(1200 * natural_b, 2))
print("enrichment:", round(p_b / natural_b, 2), " groups at the cap:", sum(math.sqrt(n_clean / n) >= 5 for n in ms4_counts))

## Q4. Warmup and cosine schedule, and the stepping bug

In [ ]:
# 60 linear warmup steps, then cosine decay over the next 420 steps
def lr_mult(t):
    return t / 60 if t < 60 else 0.5 * (1 + math.cos(math.pi * (t - 60) / 420))


print("lr at steps 15, 270, 400:", [f"{3e-4 * lr_mult(t):.2e}" for t in (15, 270, 400)])
# the bug: stepping once per epoch, after epoch 8 the schedule is only at t=8, still warming up
print("lr after epoch 8 if stepped once per epoch:", f"{3e-4 * lr_mult(8):.2e}")

## Q5. Early stopping trace

In [ ]:
# stop when there is no improvement bigger than min_delta for `patience` epochs
def early_stop(scores, patience, min_delta):
    best, best_ep, wait = -1, 0, 0
    for ep, s in enumerate(scores, 1):
        # a new best must beat the old best by more than min_delta to reset the counter
        if s > best + min_delta:
            best, best_ep, wait = s, ep, 0
        else:
            wait += 1
            if wait >= patience:
                return ep, best_ep, best
    # never triggered: stop epoch is None
    return None, best_ep, best


# example val scores per epoch from the question
trace = [0.512, 0.601, 0.648, 0.661, 0.664, 0.668, 0.671, 0.670, 0.672, 0.669, 0.673, 0.665]
print("min_delta 0.002 (stop, restored, score):", early_stop(trace, 3, 0.002))
print("min_delta 0 (stop, best, score):", early_stop(trace, 3, 0.0))

## Q6. Memory with mixed precision, and gradient accumulation

In [ ]:
state_mb = 25_000_000 * 4 * 4 / 1e6   # weights, gradients and two AdamW buffers
print("state MB:", state_mb)
# activations: 150 MB per image in fp32, about half (75) with amp
print("batch 16 MB, fp32 and amp:", state_mb + 16 * 150, state_mb + 16 * 75)
print("largest batch in 3400 MB, fp32 and amp:", int((3400 - state_mb) // 150), int((3400 - state_mb) // 75))
# accumulation: 4 micro batches of 8 = effective batch 32, loss divided by 4 before backward
print("effective batch:", 8 * 4, " micro batches:", 960 // 8, " steps:", 960 // 8 // 4, " loss divided by:", 4, " warmup steps:", 2 * (960 // 8 // 4))

## Q7. The naive Dice trap

In [ ]:
# predicting all empty gets every empty pair right but dice 0 on defects
ms4_empty = (ms4_sub.mask_rle.str.strip() == "").sum()
print("empty pairs:", ms4_empty, " baseline naive Dice:", round(ms4_empty / len(ms4_sub), 4), " baseline official:", 0.7 * 0 + 0.2 * 1 + 0.1 * 1)
# toy example from the question: official score vs naive dice
toy_fp = 1 - (34 * 1024 / CANVAS) / 340
print("toy official:", round(0.7 * 0.5 + 0.2 * toy_fp + 0.1 * 306 / 340, 4), " toy naive:", round((60 * 0.5 + 306) / 400, 4), " baseline naive:", round(340 / 400, 4))

## Q9. Defect size bins

In [ ]:
# area of every non-empty pair, then split into small / medium / large bins
pair_area = np.array([ms4_decode(r).sum() for r in ms4_sub.mask_rle[ms4_sub.mask_rle.str.strip() != ""]])
small, medium, large = pair_area[pair_area <= 100], pair_area[(pair_area > 100) & (pair_area <= 500)], pair_area[pair_area > 500]
# expected missed pairs using the miss rates from the question
missed = {"small": 0.7 * len(small), "medium": 0.3 * len(medium), "large": 0.05 * len(large)}
print("pairs per bin:", len(small), len(medium), len(large))
print("share of pixels in 501+ (%):", round(100 * large.sum() / pair_area.sum(), 2))
print("missed pairs:", missed, " total:", round(sum(missed.values()), 2))

## Q8 and Q10. Six epoch runs (R0, R0b, R1 and R2)

R1 is R0 plus warmup and cosine, and it is also the run for Q8. R2 is R0 plus the sampler. R0b only changes the seed, to estimate noise.

In [ ]:
# rare combos (count 1) cannot be stratified, they go to train
ms4_group = ms4_combo.where(ms4_combo.map(ms4_counts) >= 2, "rare_other")
ms4_rest = ms4_group[ms4_group != "rare_other"]
# 240 val images, stratified on the combination
ms4_tr, ms4_va = train_test_split(ms4_rest.index.to_numpy(), test_size=240, random_state=42, stratify=ms4_rest.values)
ms4_tr = np.concatenate([ms4_tr, ms4_group.index[ms4_group == "rare_other"].to_numpy()])
print("split:", len(ms4_tr), len(ms4_va))

# cache all 1200 images and masks in memory to speed up the 4 runs
MS4_IMG = {i: cv2.imread(str(TRAIN_DIR / f"{i}.jpg"), cv2.IMREAD_GRAYSCALE) for i in ms4_ids}
MS4_MSK = {i: np.stack([ms4_decode(ms4_wide.loc[i, c]) for c in CLASSES]).astype(np.uint8) for i in ms4_ids}


class SubsetData(Dataset):
    def __init__(self, image_ids):
        self.ids = list(image_ids)

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, k):
        i = self.ids[k]
        return torch.from_numpy(MS4_IMG[i][None].astype(np.float32) / 255.0), torch.from_numpy(MS4_MSK[i].astype(np.float32))


# validation: loss + official style score parts at threshold 0.5
def ms4_eval(model, criterion):
    model.eval()
    loss_sum, dice_sum, n_pos, fp_sum, ok, n_emp = 0.0, 0.0, 0, 0.0, 0, 0
    with torch.inference_mode():
        for x, y in DataLoader(SubsetData(ms4_va), batch_size=16):
            x, y = x.to(DEVICE), y.to(DEVICE)
            with torch.amp.autocast("cuda"):
                logits = model(x)
                loss_sum += criterion(logits, y).item() * x.size(0)
            pred = torch.sigmoid(logits.float()) >= 0.5
            truth = y > 0.5
            # per (image, class) pixel counts
            pp, tp = pred.sum((2, 3)).float(), truth.sum((2, 3)).float()
            inter = (pred & truth).sum((2, 3)).float()
            pos = tp > 0
            # dice on defect pairs, fp pixels and empty accuracy on empty pairs
            dice_sum += (2 * inter / (pp + tp).clamp(min=1))[pos].sum().item()
            n_pos += pos.sum().item()
            fp_sum += (pp[~pos] / CANVAS).sum().item()
            ok += (pp[~pos] == 0).sum().item()
            n_emp += (~pos).sum().item()
    # combine into the official 0.7 / 0.2 / 0.1 score
    fg, fps, ea = dice_sum / n_pos, 1 - fp_sum / n_emp, ok / n_emp
    return {"val_loss": loss_sum / len(ms4_va), "fg_dice": fg, "fp_score": fps, "empty_acc": ea,
            "score": 0.7 * fg + 0.2 * fps + 0.1 * ea, "false_alarm_pct": 100 * (1 - ea)}


MS4_EPOCHS = 6


# one run: R0 is the base, cosine=True adds warmup+cosine, sampler=True adds the sqrt sampler
def ms4_run(name, seed, cosine=False, sampler=False):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    gen = torch.Generator().manual_seed(seed)
    model = smp.Unet("resnet34", encoder_weights="imagenet", in_channels=1, classes=4).to(DEVICE)
    criterion = CompoundLoss()
    opt = torch.optim.AdamW(model.parameters(), lr=3e-4)
    scaler = torch.amp.GradScaler("cuda")
    ds = SubsetData(ms4_tr)
    if sampler:
        # bounded sqrt weights per training image, sampled with replacement
        tr_groups = ms4_combo.loc[ms4_tr]
        tr_counts = tr_groups.value_counts()
        w = [min(math.sqrt(tr_counts["clean"] / tr_counts[g]), 5) for g in tr_groups]
        loader = DataLoader(ds, batch_size=16, sampler=WeightedRandomSampler(w, len(ds), replacement=True, generator=gen))
    else:
        loader = DataLoader(ds, batch_size=16, shuffle=True, generator=gen)
    # warmup for 1 epoch then cosine to the end, counted in steps (batches)
    steps = len(loader)
    total = MS4_EPOCHS * steps

    def mult(t):
        return t / steps if t < steps else 0.5 * (1 + math.cos(math.pi * (t - steps) / (total - steps)))

    # lambdalr multiplies the base lr 3e-4 by mult(step)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, mult) if cosine else None
    wb = wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY, name=name, mode=WANDB_MODE, reinit=True,
                    config={"seed": seed, "cosine": cosine, "sampler": sampler, "epochs": MS4_EPOCHS, "milestone": 4})
    rows = []
    for epoch in range(1, MS4_EPOCHS + 1):
        model.train()
        s, n = 0.0, 0
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            # amp: autocast forward, scaled backward, scaler step
            opt.zero_grad()
            with torch.amp.autocast("cuda"):
                loss = criterion(model(x), y)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            # step the scheduler every batch, not every epoch (that was the bug)
            if sched:
                sched.step()
            s += loss.item() * x.size(0)
            n += x.size(0)
        # validate and log each epoch
        row = {"run": name, "epoch": epoch, "train_loss": s / n, **ms4_eval(model, criterion)}
        rows.append(row)
        wandb.log(row)
        print(name, epoch, {k: round(v, 4) for k, v in row.items() if k not in ("run", "epoch")}, flush=True)
    wb.finish()
    # free gpu memory before the next run
    del model, opt
    gc.collect()
    torch.cuda.empty_cache()
    return rows

In [ ]:
# R0 base, R0b only changes the seed, R1 adds cosine, R2 adds the sampler
ms4_rows = []
for name, seed, c, s in [("M4_R0", 42, False, False), ("M4_R0b", 7, False, False), ("M4_R1_Q8", 42, True, False), ("M4_R2", 42, False, True)]:
    ms4_rows += ms4_run(name, seed, c, s)
ms4_table = pd.DataFrame(ms4_rows)
ms4_table.to_csv("/kaggle/working/m4_assessment_runs.csv", index=False)
# best epoch of each run by score
best = ms4_table.loc[ms4_table.groupby("run").score.idxmax()].set_index("run")
print(best[["epoch", "score", "false_alarm_pct"]].round(3))
# R0 vs R0b difference = noise, a gain smaller than this is not real
noise = abs(best.score["M4_R0"] - best.score["M4_R0b"])
print("noise band:", round(noise, 3), " gain R1:", round(best.score["M4_R1_Q8"] - best.score["M4_R0"], 3), " gain R2:", round(best.score["M4_R2"] - best.score["M4_R0"], 3))
# Q8: epoch with the lowest val loss vs epoch with the best score
q8 = ms4_table[ms4_table.run == "M4_R1_Q8"]
print("Q8 lowest val loss epoch:", int(q8.loc[q8.val_loss.idxmin(), "epoch"]), " best score epoch:", int(q8.loc[q8.score.idxmax(), "epoch"]), round(q8.score.max(), 3))
print("Q8 gap at the last epoch:", round(q8.val_loss.iloc[-1] - q8.train_loss.iloc[-1], 3))
del MS4_IMG, MS4_MSK
gc.collect()

---
# Milestone 4: longer training, ensemble and final submission

The Milestone 3 model C was still improving at epoch 25, so here I train two EfficientNet-B3 FPN models for 40 epochs.
Changes compared to Milestone 3:
- images are kept in memory as uint8 (faster epochs), flips and a small brightness/contrast change are done on the GPU
- images with defect_a and defect_b are drawn more often (x2 and x3) because these classes are rare
- model weights are averaged over the training steps (EMA)
- the second model uses another seed and a larger BCE weight on defect_b
- the tuning uses the official metric: 70% Dice on pairs with a defect, 20% false positive score, 10% empty accuracy

In [ ]:
import copy

class Gray(Dataset):
    # reads the image once as uint8 so that the whole train set fits in memory
    def __init__(self, df, folder):
        self.ds = SurfaceAnomalyDataset(df, folder)
    def __len__(self):
        return len(self.ds)
    def __getitem__(self, i):
        x, y = self.ds[i]
        # keep one channel (image is grayscale), stored as 0-255 uint8
        return (x[:1] * 255).round().to(torch.uint8), y.to(torch.uint8)


# load the whole split once into uint8 tensors (N, 1, H, W) and (N, 4, H, W)
def load_all(df, folder):
    X = torch.empty((len(df), 1, H, W), dtype=torch.uint8)
    Y = torch.empty((len(df), 4, H, W), dtype=torch.uint8)
    i = 0
    for x, y in DataLoader(Gray(df, folder), batch_size=64, shuffle=False, num_workers=3):
        X[i:i + len(x)] = x
        Y[i:i + len(x)] = y
        i += len(x)
    return X, Y


Xtr, Ytr = load_all(tr_df, TRAIN_DIR)
Xva, Yva = load_all(va_df, TRAIN_DIR)
print("train", tuple(Xtr.shape), "val", tuple(Xva.shape))

# images with rare classes are drawn more often
has_cls = torch.cat([(Ytr[i:i + 500].amax((2, 3)) > 0) for i in range(0, len(Ytr), 500)]).numpy()
# weight 1 by default, 2 if it has defect_a, 3 if defect_b (b overrides a)
weights = np.ones(len(Ytr))
weights[has_cls[:, 0]] = 2.0
weights[has_cls[:, 1]] = 3.0
# normalise to probabilities for torch.multinomial
weights = torch.tensor(weights / weights.sum())

### Tuning helpers (official style score)

For every class I try a threshold and a minimum area on the validation split. The score of a class is added up
with the global counts, so the classes can be tuned one by one.

In [ ]:
# thresholds 0.2 to 0.8 in steps of 0.05, and the min areas to try
M4_TAUS = [round(t, 2) for t in np.arange(0.2, 0.85, 0.05)]
M4_AREAS = [0, 50, 100, 200, 300, 500, 750, 1000, 1500, 2000, 3000, 4000, 6000, 8000]


def tta4(model, x):
    # average over no flip, horizontal flip, vertical flip and both
    outs = []
    with torch.no_grad(), torch.amp.autocast("cuda", enabled=amp_on):
        for dims in ([], [3], [2], [2, 3]):
            xx = torch.flip(x, dims=dims) if dims else x
            p = torch.sigmoid(model(xx)).float()
            outs.append(torch.flip(p, dims=dims) if dims else p)
    return sum(outs) / 4


# val batches: grayscale copied to 3 channels for the imagenet encoders
def val_batches(bs=16):
    for i in range(0, len(Xva), bs):
        yield Xva[i:i + bs].float().expand(-1, 3, -1, -1) / 255.0, Yva[i:i + bs]


def val_stats(prob_fn):
    # per image and class: predicted pixels and overlap for each threshold
    # taus shaped (1, 1, T, 1, 1) so all thresholds are checked in one compare
    taus = torch.tensor(M4_TAUS, device=DEVICE).view(1, 1, -1, 1, 1)
    pcs, ics, gts = [], [], []
    for x, y in val_batches():
        x, y = x.to(DEVICE), y.to(DEVICE)
        # p has shape (B, 4, T, H, W)
        p = prob_fn(x).unsqueeze(2) >= taus
        pcs.append(p.sum((3, 4)).cpu())
        ics.append((p & (y > 0).unsqueeze(2)).sum((3, 4)).cpu())
        gts.append(y.sum((2, 3)).cpu())
    return {"pc": torch.cat(pcs).numpy(), "ic": torch.cat(ics).numpy(), "gt": torch.cat(gts).numpy()}


# one class at one (tau, area): dice sum on defect pairs, empty pairs kept empty, fp pixels on empty pairs
def class_score(st, c, tau, area):
    ti = M4_TAUS.index(tau)
    pc, ic, gt = st["pc"][:, c, ti], st["ic"][:, c, ti], st["gt"][:, c]
    # a mask with fewer pixels than area is dropped
    keep = pc >= max(area, 1)
    dice = np.where(keep, 2 * ic / np.maximum(pc + gt, 1), 0.0)
    pos = gt > 0
    return dice[pos].sum(), (~keep[~pos]).sum(), (pc * keep)[~pos].sum()


# official score: 0.7 dice + 0.2 (1 - fp pixel fraction) + 0.1 empty accuracy
def official_score(st, cfg):
    gt = st["gt"]
    n_pos, n_empty = (gt > 0).sum(), (gt == 0).sum()
    d = e = f = 0.0
    for c, name in enumerate(CLASSES):
        a, b, g = class_score(st, c, *cfg[name])
        d += a; e += b; f += g
    d, e, f = d / n_pos, e / n_empty, f / (n_empty * H * W)
    return {"dice_pos": d, "empty_acc": e, "fp_pixel": f, "score": 0.7 * d + 0.2 * (1 - f) + 0.1 * e}


# pick the best (tau, area) per class, each class adds its own part to the score
def tune_cfg(st):
    gt = st["gt"]
    n_pos, n_empty = (gt > 0).sum(), (gt == 0).sum()
    cfg = {}
    for c, name in enumerate(CLASSES):
        best = None
        for tau in M4_TAUS:
            for area in M4_AREAS:
                a, b, g = class_score(st, c, tau, area)
                s = 0.7 * a / n_pos - 0.2 * g / (n_empty * H * W) + 0.1 * b / n_empty
                if best is None or s > best[0]:
                    best = (s, tau, area)
        cfg[name] = best[1:]
    return cfg

In [ ]:
# note: this name replaces the train_long table, ms4_df keeps a copy
def train_long(name, arch, enc, epochs, lr, seed, pos_weight, notes):
    torch.manual_seed(seed)
    # fpn or unet from smp with imagenet encoder, wrapped with imagenet normalisation
    net = Normalized({"fpn": smp.FPN, "unet": smp.Unet}[arch](enc, encoder_weights="imagenet", in_channels=3, classes=4)).to(DEVICE)
    # ema copy of the weights, used for validation and saving
    ema = copy.deepcopy(net).eval()
    # pos_weight per class, shape (1, 4, 1, 1) to broadcast in bce
    pw = torch.tensor(pos_weight, device=DEVICE).view(1, 4, 1, 1)

    # bce with pos_weight + dice
    def loss_fn(logits, target):
        bce = nn.functional.binary_cross_entropy_with_logits(logits, target, pos_weight=pw)
        prob = torch.sigmoid(logits)
        dims = (0, 2, 3)
        dice = (2 * (prob * target).sum(dims) + 1) / (prob.sum(dims) + target.sum(dims) + 1)
        return bce + (1 - dice.mean())

    run = wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY, name=name, mode=WANDB_MODE, reinit=True,
                     config={"model": name, "encoder": enc, "epochs": epochs, "lr": lr, "batch_size": 16, "loss": "bce+dice",
                             "augment": "hflip+vflip+brightness/contrast", "seed": seed, "pos_weight": pos_weight, "notes": notes})
    opt = torch.optim.AdamW(net.parameters(), lr=lr, weight_decay=1e-4)
    # fixed number of steps per epoch, onecycle stepped every batch
    steps = len(Xtr) // 16
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=epochs * steps, pct_start=0.1)
    scaler = torch.amp.GradScaler("cuda", enabled=amp_on)
    # fixed tau 0.4 and area 100, only to track progress during training
    fixed = {c: (0.4, 100) for c in CLASSES}
    history = []
    for epoch in range(1, epochs + 1):
        start = time.time()
        net.train()
        # weighted sampling with replacement, rare class images drawn more often
        idx = torch.multinomial(weights, steps * 16, replacement=True)
        running = 0.0
        for s in range(steps):
            ib = idx[s * 16:(s + 1) * 16]
            # uint8 -> float in [0, 1], gray copied to 3 channels
            x = Xtr[ib].to(DEVICE).float().expand(-1, 3, -1, -1) / 255.0
            y = Ytr[ib].to(DEVICE).float()
            # random flips, brightness and contrast for each image
            n = x.size(0)
            fh = (torch.rand(n, device=DEVICE) < 0.5).view(-1, 1, 1, 1)
            fv = (torch.rand(n, device=DEVICE) < 0.5).view(-1, 1, 1, 1)
            x = torch.where(fh, torch.flip(x, [3]), x); y = torch.where(fh, torch.flip(y, [3]), y)
            x = torch.where(fv, torch.flip(x, [2]), x); y = torch.where(fv, torch.flip(y, [2]), y)
            # contrast and brightness jitter, only on the image
            contrast = 1 + (torch.rand(n, 1, 1, 1, device=DEVICE) - 0.5) * 0.3
            bright = (torch.rand(n, 1, 1, 1, device=DEVICE) - 0.5) * 0.16
            x = ((x - 0.5) * contrast + 0.5 + bright).clamp(0, 1)
            # amp forward, scaled backward, optimizer step and scheduler step
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", enabled=amp_on):
                loss = loss_fn(net(x), y)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            sched.step()
            running += loss.item()
            # moving average of the weights
            decay = min(0.998, (1 + epoch * steps) / (10 + epoch * steps))
            with torch.no_grad():
                for pe, pn in zip(ema.state_dict().values(), net.state_dict().values()):
                    if pe.dtype.is_floating_point:
                        pe.mul_(decay).add_(pn.detach(), alpha=1 - decay)
                    else:
                        pe.copy_(pn)
        # score the ema model on val after each epoch
        sc = official_score(val_stats(lambda xb: torch.sigmoid(ema(xb)).float()), fixed)
        row = {"epoch": epoch, "train_loss": running / steps, **sc, "lr": sched.get_last_lr()[0]}
        history.append(row)
        wandb.log(row)
        print(f"{name} epoch {epoch}: loss {row['train_loss']:.4f} dice_pos {sc['dice_pos']:.4f} "
              f"empty {sc['empty_acc']:.4f} score {sc['score']:.4f} ({time.time() - start:.0f}s)", flush=True)
    # save the final ema weights (no best epoch pick here)
    torch.save(ema.state_dict(), f"/kaggle/working/m4_{name}.pt")
    run.summary["final_val_score"] = history[-1]["score"]
    run.finish()
    pd.DataFrame(history).to_csv(f"/kaggle/working/m4_{name}_history.csv", index=False)
    return ema.eval()

### Model D: FPN with EfficientNet-B3 encoder, 40 epochs

In [ ]:
# all pos_weights 1, so plain bce + dice
model_d = train_long("D_effb3_fpn", "fpn", "efficientnet-b3", 40, 5e-4, SEED, [1.0, 1.0, 1.0, 1.0],
                     "longer training, oversampling of rare classes, EMA")

### Model E: same model, other seed and a larger BCE weight on defect_b

In [ ]:
# pos_weight 8 on defect_b to push recall for that class
model_e = train_long("E_effb3_fpn_seed7", "fpn", "efficientnet-b3", 40, 5e-4, 7, [1.0, 8.0, 1.0, 1.0],
                     "other seed, BCE weight 8 on defect_b")

### Ensemble D + E, tuning and final submission

Probabilities of D and E (flip test time augmentation over 4 flips) are averaged. The threshold and minimum area of each
class are chosen on the validation split with the official metric. To see how much this tuning overfits, I also tune on one half of
the validation images and score on the other half.

In [ ]:
m4_models = {"D": model_d, "E": model_e}
for m in m4_models.values():
    m.eval()
# tta4 probs for D, E and the D+E average, then tune tau and area per class
m4_stats = {}
for combo in (["D"], ["E"], ["D", "E"]):
    # c=combo keeps the current combo inside the lambda
    m4_stats["+".join(combo)] = val_stats(lambda x, c=combo: sum(tta4(m4_models[k], x) for k in c) / len(c))
m4_cfg = {}
for key, st in m4_stats.items():
    m4_cfg[key] = tune_cfg(st)
    r = official_score(st, m4_cfg[key])
    print(f"{key}: official style val score {r['score']:.4f} (dice on defect pairs {r['dice_pos']:.4f}, empty accuracy {r['empty_acc']:.4f})")

# check the tuning does not overfit: tune on even images, score on odd, and the other way
st = m4_stats["D+E"]
half_a, half_b = np.arange(0, len(st["gt"]), 2), np.arange(1, len(st["gt"]), 2)
sub_st = lambda idx: {k: v[idx] for k, v in st.items()}
print("tuned on half A, scored on half B:", round(official_score(sub_st(half_b), tune_cfg(sub_st(half_a)))["score"], 4))
print("tuned on half B, scored on half A:", round(official_score(sub_st(half_a), tune_cfg(sub_st(half_b)))["score"], 4))
# final config = tuned on the full val split for D+E
final_cfg = m4_cfg["D+E"]
for name in CLASSES:
    print(name, "threshold", final_cfg[name][0], "min_area", final_cfg[name][1])

---
# Milestone 5: post-processing, TTA, ensembling and the final pipeline

The worked questions are calculated below. Then I check each step of my final pipeline on the validation split, to see what it adds.

## Q1 and Q2. Thresholds

In [ ]:
logits5 = np.array([-2.0, -0.9, -0.5, 0.0, 0.8, 2.2])
truth5 = np.array([0, 0, 1, 1, 1, 0])
# sigmoid turns logits into probabilities
probs5 = 1 / (1 + np.exp(-logits5))


def dice5(pred):
    return 2 * (pred & (truth5 == 1)).sum() / (pred.sum() + truth5.sum())


# T=0.3 in logit space is log(0.3/0.7); > vs >= only matters at exactly 0.5
print("logit for T=0.3:", round(math.log(0.3 / 0.7), 4), " strict and >= at 0.5:", (probs5 > 0.5).sum(), (probs5 >= 0.5).sum())
print("Dice at 0.3 and 0.5:", round(dice5(probs5 > 0.3), 4), round(dice5(probs5 > 0.5), 4))
# official score of the three configs from the question
for n, (f, p, e) in {"X": (0.62, 0.995, 0.40), "Y": (0.58, 0.999, 0.85), "Z": (0.45, 0.9998, 0.97)}.items():
    print("config", n, round(0.7 * f + 0.2 * p + 0.1 * e, 4))

## Q3, Q4 and Q5. Connected components and area filtering

In [ ]:
grid5 = np.array([[1, 1, 0, 0, 0, 0, 0, 1], [1, 1, 0, 0, 0, 0, 0, 0], [0, 0, 1, 0, 0, 0, 0, 0],
                  [0, 0, 0, 1, 0, 0, 0, 0], [0, 0, 0, 0, 0, 1, 1, 0], [1, 0, 0, 0, 0, 1, 1, 1]], dtype=np.uint8)
# 8-connectivity joins diagonal pixels, 4-connectivity does not
n8, _, stats8, _ = cv2.connectedComponentsWithStats(grid5, connectivity=8)
n4, _, _, _ = cv2.connectedComponentsWithStats(grid5, connectivity=4)
# component areas (label 0 is the background, skipped)
a8 = stats8[1:, cv2.CC_STAT_AREA]
print("components 8 and 4:", n8 - 1, n4 - 1, " pixels kept at N=5 and N=6:", a8[a8 >= 5].sum(), a8[a8 >= 6].sum())

# question numbers: score parts before and after the area filter
fg_b, fp_b, ea_b = (12 * 0.7 + 8 * 0.45) / 20, 1 - (20 * 20 + 10 * 2000) / CANVAS / 80, 50 / 80
fg_a, fp_a, ea_a = (12 * 0.7 + 7 * 0.45) / 20, 1 - (10 * 2000) / CANVAS / 80, 70 / 80
print("after:", round(fg_a, 4), round(fp_a, 4), round(ea_a, 4), " score before and after:", round(0.7 * fg_b + 0.2 * fp_b + 0.1 * ea_b, 4), round(0.7 * fg_a + 0.2 * fp_a + 0.1 * ea_a, 4))

# count real masks that a min component size of 100 would remove completely
wiped = {c: 0 for c in CLASSES}
for row in ms4_df[ms4_df.mask_rle.str.strip() != ""].itertuples():
    _, _, stats, _ = cv2.connectedComponentsWithStats(ms4_decode(row.mask_rle), connectivity=8)
    if stats[1:, cv2.CC_STAT_AREA].max() < 100:
        wiped[row.defect_type] += 1
print("pairs wiped out at N=100:", sum(wiped.values()), wiped)

## Q6 and Q7. TTA and ensembling

In [ ]:
orig5 = np.array([[0.9, 0.6], [0.4, 0.2]])
h5, v5 = np.array([[0.5, 0.8], [0.3, 0.4]]), np.array([[0.1, 0.5], [0.7, 0.9]])
# flip each tta prediction back before averaging
right = (orig5 + h5[:, ::-1] + v5[::-1, :]) / 3
# bug: averaging without flipping back mixes different pixels
wrong = (orig5 + h5 + v5) / 3
print("flip dims on (B, C, H, W): horizontal 3, vertical 2")
print("top right:", round(right[0, 1], 4), " positive correct and buggy:", (right > 0.5).sum(), (wrong.round(6) > 0.5).sum())

# three models' probs for 5 pixels, t5 is the truth
A5, B5, C5 = np.array([0.9, 0.55, 0.8, 0.1, 0.6]), np.array([0.4, 0.55, 0.7, 0.2, 0.45]), np.array([0.45, 0.2, 0.9, 0.3, 0.55])
t5 = np.array([1, 0, 1, 0, 1]).astype(bool)
d5 = lambda p: round(2 * (p & t5).sum() / (p.sum() + t5.sum()), 4)
# majority vote vs soft (probability) averaging
votes = (A5 > 0.5).astype(int) + (B5 > 0.5) + (C5 > 0.5)
sig = lambda x: 1 / (1 + math.exp(-x))
print("Dice soft and vote:", d5((A5 + B5 + C5) / 3 > 0.5), d5(votes >= 2), " weighted p1:", round(0.5 * 0.9 + 0.3 * 0.4 + 0.2 * 0.45, 4))
# probability averaging vs logit averaging for logits 1 and -3
print("probability average and logit average:", round((sig(1) + sig(-3)) / 2, 4), round(sig(-1), 4))

## Q8. Inference budget

In [ ]:
# size of 5000 images x 4 classes x 128 x 800 probability maps
map_b = 5000 * 4 * 128 * 800
print("GB for float32, float16, uint8:", map_b * 4 / 1e9, map_b * 2 / 1e9, map_b / 1e9)
# a running sum needs only one set of maps, not one per view
print("naive GB, config A and B:", 9 * map_b * 4 / 1e9, 4 * map_b * 4 / 1e9, " running sum GB:", map_b * 4 / 1e9)
# time = number of passes x 0.012 s per pass
print("passes A and B:", 9 * 5000, 4 * 5000, " minutes:", 9 * 5000 * 0.012 / 60, 4 * 5000 * 0.012 / 60)
print("most models in 22 minutes with 3 and 2 views:", int(22 // (3 * 5000 * 0.012 / 60)), int(22 // (2 * 5000 * 0.012 / 60)))

## What each step of my pipeline adds (validation)

One change at a time, with the thresholds and minimum areas of my final config.
My final pipeline order is: image, each model with flip TTA (each output flipped back), average of probabilities, threshold, area filter, RLE.

In [ ]:
def one_view(model):
    return lambda x: torch.sigmoid(model(x)).float()


# add one step at a time: no tta, 4 flip tta, other model, ensemble
ladder = {"D, no TTA": one_view(model_d), "D, 4 flip TTA": lambda x: tta4(model_d, x),
          "E, 4 flip TTA": lambda x: tta4(model_e, x), "D + E, 4 flip TTA": lambda x: (tta4(model_d, x) + tta4(model_e, x)) / 2}
ladder_rows = {}
# same final_cfg for all, so only the probabilities change
with torch.no_grad():
    for name, fn in ladder.items():
        ladder_rows[name] = official_score(val_stats(fn), final_cfg)
print(pd.DataFrame(ladder_rows).T.round(4))

## Connected component filter

My final config drops a mask only when its whole area is small. The milestone also covers removing each small blob. Here I compare the two on validation.
`COMPONENT_N` stays `None`, so the final submission does not change. If the validation gain looks real, it can be set to `cc_best` for a later submission.

In [ ]:
def remove_small(mask, n):
    # keeps only the 8-connected components with area of at least n pixels
    if n <= 0:
        return mask
    # label the components, label 0 is the background
    count, labels, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
    # keep only labels with area >= n, then map labels back to a mask
    keep = np.zeros(count, dtype=np.uint8)
    keep[1:] = stats[1:, cv2.CC_STAT_AREA] >= n
    return keep[labels]


# None = component filter is off in the final submission
COMPONENT_N = None
cc_sizes = [0, 25, 50, 100, 200, 500]
# per class and size: [dice sum on defect pairs, empty pairs kept empty, fp pixels]
cc = {c: {n: [0.0, 0, 0.0] for n in cc_sizes} for c in CLASSES}
n_pos_v = n_emp_v = 0
with torch.no_grad():
    for x, y in val_batches():
        # D + E ensemble with 4 flip tta
        p = ((tta4(model_d, x.to(DEVICE)) + tta4(model_e, x.to(DEVICE))) / 2).cpu().numpy()
        y = y.numpy()
        for i in range(len(p)):
            for k, name in enumerate(CLASSES):
                truth = y[i, k] > 0
                pos = truth.any()
                n_pos_v += pos
                n_emp_v += not pos
                # threshold with the final tau, then try each component size
                raw = (p[i, k] >= final_cfg[name][0]).astype(np.uint8)
                for n in cc_sizes:
                    m = remove_small(raw, n).astype(bool)
                    if pos:
                        cc[name][n][0] += 2 * (m & truth).sum() / max(m.sum() + truth.sum(), 1)
                    else:
                        cc[name][n][1] += not m.any()
                        cc[name][n][2] += m.sum()
# class part of the score (the 0.2 constant is added once below)
class_value = lambda name, n: 0.7 * cc[name][n][0] / n_pos_v + 0.1 * cc[name][n][1] / n_emp_v - 0.2 * cc[name][n][2] / (n_emp_v * CANVAS)
# best component size per class on val
cc_best = {name: max(cc_sizes, key=lambda n: class_value(name, n)) for name in CLASSES}
cc_score = 0.2 + sum(class_value(name, cc_best[name]) for name in CLASSES)
print("component filter sizes:", cc_best, " validation score:", round(cc_score, 4))
print("current whole mask rule, validation score:", round(official_score(m4_stats["D+E"], final_cfg)["score"], 4))

### Final submission from D + E

In [ ]:
test_ids = sample.image_id.drop_duplicates().tolist()


# test images as rgb (3, H, W) in [0, 1], same input as training
class TestDataset(Dataset):
    def __init__(self, ids):
        self.ids = ids

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        image = cv2.imread(str(find_image(TEST_DIR, self.ids[idx])), cv2.IMREAD_COLOR)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
        return torch.from_numpy(image.transpose(2, 0, 1).copy()), self.ids[idx]


test_loader4 = DataLoader(TestDataset(test_ids), batch_size=16, shuffle=False, num_workers=4)
prediction = {}
# D + E average with 4 flip tta, then per class threshold
for x, ids in test_loader4:
    x = x.to(DEVICE)
    p = (tta4(model_d, x) + tta4(model_e, x)).cpu().numpy() / 2
    for i, image_id in enumerate(ids):
        for c, name in enumerate(CLASSES):
            tau, area = final_cfg[name]
            m = (p[i, c] >= tau).astype(np.uint8)
            # component filter only if COMPONENT_N was set
            if COMPONENT_N:
                m = remove_small(m, COMPONENT_N[name])
            # drop the mask if it has fewer pixels than the tuned min area
            prediction[(image_id, name)] = rle_encode(m) if m.sum() >= max(area, 1) else ""

submission = sample.copy()
submission["mask_rle"] = [prediction[(i, c)] for i, c in zip(submission.image_id, submission.defect_type)]
# empty mask is written as a single space like sample_submission.csv
submission["mask_rle"] = submission["mask_rle"].replace("", " ")
assert list(submission.columns) == ["ID", "image_id", "defect_type", "mask_rle"]
assert len(submission) == len(sample) and submission.ID.equals(sample.ID)
submission.to_csv("/kaggle/working/submission.csv", index=False)
print("non-empty predictions per class:")
print(submission[submission.mask_rle != " "].defect_type.value_counts())
print("Saved /kaggle/working/submission.csv")